# 02 — Time-Domain Analysis
**Phase 1 · Multi-fault Diagnosis of Rotating Machinery**

Goals:
- Extract time-domain features per accelerometer channel (RMS, kurtosis, crest factor, etc.)
- Compare feature distributions across fault conditions
- Identify the most discriminative time-domain features

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.fft import rfft, rfftfreq

In [ ]:
DATA_ROOT = Path("../../data/raw")
SAMPLING_RATE = 50_000  # Hz

CHANNEL_NAMES = [
    "tachometer",
    "underhang_axial", "underhang_radial", "underhang_tangential",
    "overhang_axial",  "overhang_radial",  "overhang_tangential",
    "microphone",
]

ACCEL_CHANNELS = [
    "underhang_axial", "underhang_radial", "underhang_tangential",
    "overhang_axial",  "overhang_radial",  "overhang_tangential",
]

In [ ]:
# TODO: import or redefine time_domain_features() and estimate_rotation_frequency()
# Migrate from eda-mafaulda.ipynb once stabilised

In [ ]:
# TODO: run feature extraction across all files and store in a DataFrame

In [ ]:
# TODO: boxplots / violin plots of each feature grouped by fault class

In [ ]:
def estimate_rotation_frequency(tacho: np.ndarray, fs: int = SAMPLING_RATE) -> float:
    tacho = np.asarray(tacho, dtype=np.float64) - tacho.mean()
    mag   = np.abs(rfft(tacho * np.hanning(len(tacho))))
    freqs = rfftfreq(len(tacho), d=1.0 / fs)
    valid = (freqs >= 5.0) & (freqs <= 100.0)
    return float(freqs[valid][np.argmax(mag[valid])]) if valid.any() else 0.0


def time_domain_features(x: np.ndarray) -> dict:
    x        = np.asarray(x, dtype=np.float64)
    abs_x    = np.abs(x)
    rms      = float(np.sqrt(np.mean(x ** 2)))
    peak     = float(abs_x.max())
    mean_abs = float(abs_x.mean())
    sqrt_mean_sq = float(np.mean(np.sqrt(abs_x)) ** 2)

    return {
        "mean":               float(x.mean()),
        "std":                float(x.std()),
        "rms":                rms,
        "peak":               peak,
        "peak_to_peak":       float(x.max() - x.min()),
        "crest_factor":       peak / rms          if rms          > 0 else 0.0,
        "shape_factor":       rms  / mean_abs     if mean_abs     > 0 else 0.0,
        "impulse_factor":     peak / mean_abs     if mean_abs     > 0 else 0.0,
        "margin_factor":      peak / sqrt_mean_sq if sqrt_mean_sq > 0 else 0.0,
        "skewness":           float(stats.skew(x)),
        "kurtosis":           float(stats.kurtosis(x) + 3),
        "zero_crossing_rate": float(np.sum(np.diff(np.signbit(x))) / (len(x) / SAMPLING_RATE)),
    }


In [ ]:
NORMAL_DIR = Path("/kaggle/input/datasets/vuxuancu/mafaulda-full/mafaulda/normal")

csv_files = sorted(NORMAL_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} files in normal/")

records = []
for csv_path in csv_files:
    df = pd.read_csv(csv_path, header=None, names=CHANNEL_NAMES)
    f_r = estimate_rotation_frequency(df["tachometer"].values)

    for ch in ACCEL_CHANNELS:
        feats = time_domain_features(df[ch].values)
        records.append({
            "file":             csv_path.name,
            "fault_type":       "normal",
            "rotation_freq_hz": f_r,
            "channel":          ch,
            **feats,
        })

df_normal = pd.DataFrame(records)
print(f"{df_normal.shape[0]} rows  ({len(csv_files)} files × {len(ACCEL_CHANNELS)} channels)")
df_normal.head(12)


In [ ]:
FEATURE_COLS = [
    "mean", "std", "rms", "peak", "peak_to_peak",
    "crest_factor", "shape_factor", "impulse_factor",
    "margin_factor", "skewness", "kurtosis", "zero_crossing_rate",
]

df_normal.groupby("channel")[FEATURE_COLS].agg(["mean", "std"]).round(4)
